# Milestones 1-2: Symbolic Pruning and Ollama Ranking

This notebook now implements:

- symbolic pruning and verification over relation bundles,
- LLM-ready prompt construction over verified candidates,
- an optional local Ollama ranking step that operates on the verified shortlist only.


In [ ]:
from milestone1_core import *

c1 = [
    NamedRect("pizza", Rect(-2, 3, 3, 4)),
    NamedRect("cutter", Rect(-4, 3, 1, 6)),
]

c2 = [
    NamedRect("pizza", Rect(-2, 3, 3, 4)),
    NamedRect("cutter", Rect(-5, 3, 1, 6)),
]

delta = changed_params(c1, c2)
eq_pool = detect_c1_equations(
    c1,
    include_offsets=True,
    include_pins=True,
    linear_only=True,
    print_list=True,
)

print(f"\ndelta: {tuple(sorted(delta))}")
plot_canvas(c1, bounds=(-5, 5, -5, 5), name="C1")
plot_canvas(c2, bounds=(-5, 5, -5, 5), name="C2")


## Milestone 1 Workflow

Primary funnel:

`naive subsets -> delta-overlap filter -> verifier`

Diagnostic view:

`naive subsets -> delta-overlap filter -> shared-variable count`

The shared-variable heuristic is reported, but it is **not** used as a hard rejection rule.


In [ ]:
from dataclasses import replace
import json

from milestone1_analysis import *
from milestone1_ollama import *


In [ ]:
MAX_SYSTEM_SIZE = 5
verifier_context = build_verifier_context(c1, c2, eq_pool)

bundle_stats, verified_bundles = analyze_relation_bundles(
    c1,
    c2,
    eq_pool,
    max_system_size=MAX_SYSTEM_SIZE,
    context=verifier_context,
)

print_search_report(bundle_stats, verified_bundles)
print()
print_surviving_bundle_ids(verified_bundles, limit=30)
print()

top_verified_bundles = materialize_bundle_records(
    c1,
    c2,
    eq_pool,
    verified_bundles[:8],
    context=verifier_context,
)
print_top_verified_bundles(top_verified_bundles, top_k=len(top_verified_bundles))


In [ ]:
bundle_000_014 = verify_bundle(
    c1,
    c2,
    eq_pool,
    (0, 14),
    context=verifier_context,
    include_details=True,
)
bundle_014_024 = verify_bundle(
    c1,
    c2,
    eq_pool,
    (14, 24),
    context=verifier_context,
    include_details=True,
)

print_bundle_summary(bundle_000_014)
print()
print_bundle_summary(bundle_014_024)


## Milestone 2 Workflow

This section builds a structured prompt over the verified shortlist and optionally sends it to a local Ollama model for ranking.


In [ ]:
RUN_OLLAMA_RANKING = False
OLLAMA_BASE_URL = "http://localhost:11434/api"
OLLAMA_MODEL = "<replace-with-local-model>"
OLLAMA_TOP_K = 5
OLLAMA_MAX_TIES_PER_BUNDLE = 3
USER_HISTORY_SUMMARY = "The user dragged cutter.x from -4 to -5 while leaving the rest of the scene untouched."

ollama_config = OllamaRankerConfig(
    model_name=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
)


In [ ]:
ranking_candidates = materialize_bundle_records(
    c1,
    c2,
    eq_pool,
    verified_bundles[:OLLAMA_TOP_K],
    context=verifier_context,
)
ranking_schema = build_ranking_schema(len(ranking_candidates))
ranking_prompt, ranking_candidate_lookup = build_llm_ranking_prompt(
    c1,
    c2,
    ranking_candidates,
    top_k=len(ranking_candidates),
    user_history=USER_HISTORY_SUMMARY,
    max_ties_per_bundle=OLLAMA_MAX_TIES_PER_BUNDLE,
    schema=ranking_schema,
)

print(ranking_prompt)

ollama_ready = False
ollama_model_names = []
if RUN_OLLAMA_RANKING:
    if OLLAMA_MODEL == "<replace-with-local-model>":
        raise ValueError(
            "Set OLLAMA_MODEL to a value returned by /api/tags before enabling RUN_OLLAMA_RANKING."
        )
    ollama_model_names = list_ollama_models(ollama_config)
    if OLLAMA_MODEL not in ollama_model_names:
        raise ValueError(
            f"OLLAMA_MODEL={OLLAMA_MODEL!r} is not installed. Available models: {ollama_model_names}"
        )
    ollama_ready = True
    print(f"Ollama ready with model: {OLLAMA_MODEL}")
else:
    print("Skipping Ollama readiness check because RUN_OLLAMA_RANKING is False.")


In [ ]:
ollama_ranking_result = None
if ollama_ready:
    ollama_ranking_result = rank_verified_bundles_with_ollama(
        ranking_candidates,
        USER_HISTORY_SUMMARY,
        ollama_config,
        top_k=len(ranking_candidates),
        max_ties_per_bundle=OLLAMA_MAX_TIES_PER_BUNDLE,
        c1=c1,
        c2=c2,
        prompt_text=ranking_prompt,
        candidate_lookup=ranking_candidate_lookup,
    )
    print("Ollama ranking completed.")
else:
    print("Ollama ranking skipped.")


In [ ]:
if ollama_ranking_result is None:
    print("No live Ollama ranking result to display.")
else:
    print(f"Summary: {ollama_ranking_result.summary}")
    chosen_record = ranking_candidate_lookup[ollama_ranking_result.chosen_candidate_id]
    print(f"Chosen candidate id: {ollama_ranking_result.chosen_candidate_id}")
    print(f"Chosen eq_indices: {chosen_record.eq_indices}")
    print()
    print("Ranked candidates:")
    for ranked in ollama_ranking_result.ranked_candidates:
        record = ranking_candidate_lookup[ranked.candidate_id]
        print(
            f"  rank={ranked.bundle_rank} candidate_id={ranked.candidate_id} "
            f"eq_indices={record.eq_indices} score={ranked.score} keep={ranked.keep}"
        )
        print(f"    rationale={ranked.rationale}")
    print()
    print("Raw JSON content:")
    print(ollama_ranking_result.raw_content)


In [ ]:
assert bundle_stats["naive_total"] == 83681
assert bundle_000_014.delta_hit is True
assert bundle_000_014.has_shared_variable is False
assert bundle_000_014.verification_passed is True
assert bundle_014_024.delta_hit is True
assert bundle_014_024.verification_passed is False
assert bundle_014_024.failure_reason == "contradiction after fixing edited variables"
assert all(record.verification_passed for record in verified_bundles)
assert all(
    all(len(option.extra_fixed_vars) == record.min_extra_fixed for option in record.viable_fixed_sets)
    for record in verified_bundles
)
assert "Edited parameters from C1 to C2 (delta)" in ranking_prompt
assert "User history summary" in ranking_prompt
assert "candidate_id" in ranking_prompt
assert "Output JSON schema:" in ranking_prompt
assert '"chosen_candidate_id"' in ranking_prompt

tie_heavy_record = replace(
    ranking_candidates[0],
    viable_fixed_sets=ranking_candidates[0].viable_fixed_sets * 4,
)
tie_prompt, _ = build_llm_ranking_prompt(
    c1,
    c2,
    [tie_heavy_record],
    top_k=1,
    user_history=USER_HISTORY_SUMMARY,
    max_ties_per_bundle=3,
    schema=build_ranking_schema(1),
)
assert "omitted_tied_fixed_set_count: 1" in tie_prompt

mock_content = {
    "summary": "Candidate 1 best matches the recent edit.",
    "chosen_candidate_id": 1,
    "ranked_candidates": [
        {
            "candidate_id": candidate_id,
            "bundle_rank": rank,
            "score": 100 - 5 * (rank - 1),
            "keep": rank <= 2,
            "rationale": f"Reason {rank}",
        }
        for rank, candidate_id in enumerate(sorted(ranking_candidate_lookup.keys()), start=1)
    ],
}
mock_ollama_response = {"message": {"content": json.dumps(mock_content)}}
mock_ranking_result = parse_ranking_response(
    mock_ollama_response,
    set(ranking_candidate_lookup.keys()),
    ranking_prompt,
)
assert mock_ranking_result.chosen_candidate_id == 1
assert len(mock_ranking_result.ranked_candidates) == len(ranking_candidate_lookup)
if RUN_OLLAMA_RANKING and ollama_ranking_result is not None:
    assert ollama_ranking_result.chosen_candidate_id in ranking_candidate_lookup
print("Milestone 2 checks passed.")
